In [1]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id
import pyspark.pandas as ps

import os

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [2]:
from hypex.matching import Matching
from hypex.ml.faiss import FaissNearestNeighbors
from hypex.transformers import TypeCaster
from hypex.dataset import Dataset, InfoRole, TreatmentRole, FeatureRole, TargetRole, ExperimentData, AdditionalMatchingRole, TempTargetRole
from hypex.utils import BackendsEnum
from hypex.experiments import Experiment, OnRoleExperiment
from hypex.comparators import TTest
from hypex.operators import Bias, MatchingMetrics

In [3]:
# --- 1. Настройки окружения для macOS (Важно!) ---
# На macOS иногда возникают проблемы с форком процессов Java (Executor'ы не стартуют).
# Эта переменная часто решает проблему "Connection refused" или краши при запуске local-cluster
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

# Очистка старых сессий и переменных (как у вас было)
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("✅ Существующая сессия остановлена.")
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Конфигурация Кластера ---
# Формат: local-cluster[число_воркеров, ядер_на_воркер, память_на_воркер_в_МБ]
# Мы просим 2 экзекутора, по 1 ядру, по 2 ГБ памяти каждый
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 6
MEMORY_PER_EXECUTOR_MB = 4096 

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"🚀 Запуск в режиме: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    # Память драйвера (остается у вас)
    .config("spark.driver.memory", "4g") 
    # Память экзекутора (должна соответствовать или быть меньше чем в master URL)
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "6")
    .config("spark.executor.instances", NUM_EXECUTORS)
    # Увеличиваем память под оверхед, чтобы избежать ошибок выделения памяти
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4") # Для тестов меньше дефолтных 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Проверка конфигурации ---
print(f"✅ Сессия создана.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Проверка количества экзекуторов (может занять пару секунд на старт)
import time
time.sleep(3) 
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# --- 4. Тест на распределение (Пример) ---
# Чтобы убедиться, что задача ушла на экзекуторы, а не осталась на драйвере
def print_executor_info(iterator):
    import os
    # Получаем ID экзекутора из переменных окружения процесса
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Создаем датафрейм и применяем трансформацию
df = sp_s.range(0, 10, 1, 4) # 4 партиции
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Где выполнялись задачи:")
for line in result:
    print(line)

# Не забывайте останавливать сессию в конце скрипта, так как процессы тяжелые
# sp_s.stop() 

🚀 Запуск в режиме: local-cluster[2, 6, 4096]


26/06/02 15:17:59 WARN Utils: Your hostname, eric-Katana-17-B12UCR resolves to a loopback address: 127.0.1.1; using 10.240.79.162 instead (on interface wlo1)
26/06/02 15:17:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/02 15:18:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Сессия создана.
Driver Memory Config: 4g
Executor Memory Config: 4g


📊 Активных экзекуторов (проверка через RDD): 2

🖥️ Где выполнялись задачи:
Executor ID: Driver/Local, PID: 558287
Executor ID: Driver/Local, PID: 558293
Executor ID: Driver/Local, PID: 558342
Executor ID: Driver/Local, PID: 558335


In [4]:
n = 200  # увеличьте для теста IVF-индексов
df = pd.DataFrame({
    "treatment": np.random.choice([0, 1], size=n, p=[0.6, 0.4]),
    "feat_num_1": np.random.normal(loc=10, scale=3, size=n),
    "feat_num_2": np.random.normal(loc=-2, scale=1.5, size=n),
    "target": np.random.normal(loc=100, scale=10, size=n)
})

mathed_indexes = pd.DataFrame({
    "FaissNearestNeighbors┴┴┴0": np.random.randint(0, n, n),
    "FaissNearestNeighbors┴┴┴1": np.random.randint(0, n, n)
})

In [5]:
spark_df = sp_s.createDataFrame(df)
add_df = sp_s.createDataFrame(mathed_indexes)
# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "target": TargetRole(),
}

add_roles = {
    "FaissNearestNeighbors┴┴┴0": AdditionalMatchingRole(),
    "FaissNearestNeighbors┴┴┴1": AdditionalMatchingRole()
}

dataset = Dataset(
    roles=roles,
    data=spark_df,
    backend=BackendsEnum.spark,
    session=sp_s,
)

additional_fields = Dataset(
    roles=add_roles,
    data=add_df,
    backend=BackendsEnum.spark,
    session=sp_s,
)

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [6]:
experiment_data = ExperimentData(dataset)
experiment_data.additional_fields = additional_fields

In [7]:
experiment_data.ds.tmp_roles = {"feat_num_1": TempTargetRole()}

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [8]:
experiment = OnRoleExperiment(
    executors=[
        TTest(
            grouping_role=TreatmentRole(),
            compare_by="matched_pairs",
            baseline_role=AdditionalMatchingRole(),
        )
    ],
    role=FeatureRole()
)

In [9]:
result = experiment.execute(experiment_data)

StatsTTest


/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26/06/02 15:18:14 WARN AttachDistributedSequenceExec: clean up cached RDD(128) in AttachDistributedSequenceExec(816)
26/06/02 15:18:14 WARN AttachDistributedSequenceExec: clean u

executor.key = TTest┴┴; dt = 10.0617c
StatsTTest


/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is l

executor.key = TTest┴┴; dt = 6.1846c


/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [10]:
result.analysis_tables

{'StatsTTest┴┴feat_num_1┆stats':    mean┆feat_num_1  std┆feat_num_1  count┆feat_num_1  mean┆feat_num_1_matched  std┆feat_num_1_matched  count┆feat_num_1_matched
 0         9.911649        2.877864             128.0                 9.541747                2.931943                     128.0
 1         9.808716        3.283150              72.0                10.376070                3.071872                      72.0
 
 2 rows × 6 columns,
 'StatsTTest┴┴feat_num_1':                p-value  statistic  pass
 0┆feat_num_1  0.309336   1.018655   0.0
 1┆feat_num_1  0.286109  -1.070727   0.0
 
 2 rows × 3 columns,
 'StatsTTest┴┴feat_num_2┆stats':    mean┆feat_num_2  std┆feat_num_2  count┆feat_num_2  mean┆feat_num_2_matched  std┆feat_num_2_matched  count┆feat_num_2_matched
 0        -2.165064        1.330160             128.0                -2.328589                1.259202                     128.0
 1        -1.951347        1.495196              72.0                -2.315515                1.

In [11]:
sp_s.stop()